In [3]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import os

# 1. Load your CSV
csv_file = '../data/link_invent_outputs/scored_sampling_test_protac_optimized_full.csv'
df = pd.read_csv(csv_file)

# --- RANKING STEP ---
# Sort by Active_Probability (Highest first)
df = df.sort_values(by='Active_Probability', ascending=False).reset_index(drop=True)
# --------------------

# Create an output directory
output_dir = '../data/3d_sdfs_all'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for index, row in df.iterrows():
    smiles = row['SMILES']
    prob = row['Active_Probability']
    # Define rank starting from 1
    rank = index + 1
    
    print(f"Processing Rank {rank} (Prob: {prob:.4f})...")

    mol = Chem.MolFromSmiles(smiles)
    
    if mol:
        mol = Chem.AddHs(mol)
        
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        embed_status = AllChem.EmbedMolecule(mol, params)
        
        if embed_status == 0:
            AllChem.MMFFOptimizeMolecule(mol)
            
            # Save using the new rank
            filename = os.path.join(output_dir, f"Compound_rank_{rank}.sdf")
            writer = Chem.SDWriter(filename)
            writer.write(mol)
            writer.close()
        else:
            print(f"  Error: Could not embed 3D coordinates for rank {rank}")
    else:
        print(f"  Error: Invalid SMILES for rank {rank}")

print(f"\nDone! {len(df)} compounds processed and ranked.")

Processing Rank 1 (Prob: 0.8123)...
Processing Rank 2 (Prob: 0.8121)...
Processing Rank 3 (Prob: 0.8041)...
Processing Rank 4 (Prob: 0.8018)...
Processing Rank 5 (Prob: 0.7984)...
Processing Rank 6 (Prob: 0.7971)...
Processing Rank 7 (Prob: 0.7966)...
Processing Rank 8 (Prob: 0.7965)...
Processing Rank 9 (Prob: 0.7963)...
Processing Rank 10 (Prob: 0.7963)...
Processing Rank 11 (Prob: 0.7960)...
Processing Rank 12 (Prob: 0.7948)...
Processing Rank 13 (Prob: 0.7941)...
Processing Rank 14 (Prob: 0.7933)...
Processing Rank 15 (Prob: 0.7925)...
Processing Rank 16 (Prob: 0.7923)...
Processing Rank 17 (Prob: 0.7923)...
Processing Rank 18 (Prob: 0.7920)...
Processing Rank 19 (Prob: 0.7920)...
Processing Rank 20 (Prob: 0.7908)...
Processing Rank 21 (Prob: 0.7906)...
Processing Rank 22 (Prob: 0.7894)...
Processing Rank 23 (Prob: 0.7890)...
Processing Rank 24 (Prob: 0.7890)...
Processing Rank 25 (Prob: 0.7889)...
Processing Rank 26 (Prob: 0.7888)...
Processing Rank 27 (Prob: 0.7888)...
Processing

[15:19:32] UFFTYPER: Unrecognized atom type: S_5+4 (17)


Processing Rank 701 (Prob: 0.5436)...
Processing Rank 702 (Prob: 0.5340)...
Processing Rank 703 (Prob: 0.5141)...

Done! 703 compounds processed and ranked.


In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import os

# 1. Load your CSV
csv_file = '../data/link_invent_outputs/scored_sampling_test_protac_active_full_3.csv'
df = pd.read_csv(csv_file)

# 2. Ranking & Filtering Step
# Sort by Active_Probability (Highest first) and take only the top 100
df = df.sort_values(by='Active_Probability', ascending=False).head(100).reset_index(drop=True)
# Add the Rank column (starting from 1)
df['Rank'] = df.index + 1

# Save the updated CSV with the Rank column
ranked_csv_path = '../data/link_invent_outputs/scored_sampling_test_protac_ranked.csv'
df.to_csv(ranked_csv_path, index=False)
print(f"Ranked CSV saved to: {ranked_csv_path}")

Ranked CSV saved to: ../data/link_invent_outputs/scored_sampling_test_protac_ranked.csv


In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import os

# 1. Load your CSV
csv_file = '../data/link_invent_outputs/scored_sampling_test_protac_active_full_3.csv'
df = pd.read_csv(csv_file)

# 2. Ranking & Filtering Step
# Sort by Active_Probability (Highest first) and take only the top 100
df = df.sort_values(by='Active_Probability', ascending=False).head(100).reset_index(drop=True)

# 3. Grouping Logic: Map unique Warheads (within the top 100) to Group IDs
unique_warheads = df['Warheads'].unique()
warhead_mapping = {wh: f"Group_{i+1}" for i, wh in enumerate(unique_warheads)}

# Create the base output directory
base_output_dir = '../data/3d_sdfs_all'
if not os.path.exists(base_output_dir):
    os.makedirs(base_output_dir)

# Save the mapping key for the top 100
with open(os.path.join(base_output_dir, 'warhead_groups_key_top100.txt'), 'w') as f:
    for wh, group in warhead_mapping.items():
        f.write(f"{group}: {wh}\n")

print(f"Starting processing for the top {len(df)} compounds...")

for index, row in df.iterrows():
    smiles = row['SMILES']
    prob = row['Active_Probability']
    warhead_val = row['Warheads']
    rank = index + 1
    
    # Identify the correct subfolder
    group_folder = warhead_mapping[warhead_val]
    target_dir = os.path.join(base_output_dir, group_folder)
    
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    print(f"Rank {rank}/100 -> {group_folder} (Prob: {prob:.4f})")

    mol = Chem.MolFromSmiles(smiles)
    if mol:
        mol = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        embed_status = AllChem.EmbedMolecule(mol, params)
        
        if embed_status == 0:
            AllChem.MMFFOptimizeMolecule(mol)
            
            # Save to the subfolder
            filename = os.path.join(target_dir, f"Compound_rank_{rank}.sdf")
            writer = Chem.SDWriter(filename)
            writer.write(mol)
            writer.close()
        else:
            print(f"  Error: Could not embed 3D for rank {rank}")
    else:
        print(f"  Error: Invalid SMILES for rank {rank}")

print(f"\nDone! Processed the top 100 compounds into {len(unique_warheads)} groups.")

Starting processing for the top 100 compounds...
Rank 1/100 -> Group_1 (Prob: 0.7976)
Rank 2/100 -> Group_1 (Prob: 0.7951)
Rank 3/100 -> Group_1 (Prob: 0.7951)
Rank 4/100 -> Group_1 (Prob: 0.7933)
Rank 5/100 -> Group_2 (Prob: 0.7928)
Rank 6/100 -> Group_2 (Prob: 0.7905)
Rank 7/100 -> Group_2 (Prob: 0.7884)
Rank 8/100 -> Group_2 (Prob: 0.7834)
Rank 9/100 -> Group_2 (Prob: 0.7829)
Rank 10/100 -> Group_1 (Prob: 0.7827)
Rank 11/100 -> Group_1 (Prob: 0.7814)
Rank 12/100 -> Group_1 (Prob: 0.7813)
Rank 13/100 -> Group_1 (Prob: 0.7806)
Rank 14/100 -> Group_1 (Prob: 0.7795)
Rank 15/100 -> Group_1 (Prob: 0.7795)
Rank 16/100 -> Group_1 (Prob: 0.7788)
Rank 17/100 -> Group_2 (Prob: 0.7783)


[09:58:51] UFFTYPER: Unrecognized atom type: S_5+4 (19)


Rank 18/100 -> Group_2 (Prob: 0.7780)
Rank 19/100 -> Group_2 (Prob: 0.7770)
Rank 20/100 -> Group_1 (Prob: 0.7768)
Rank 21/100 -> Group_2 (Prob: 0.7763)
Rank 22/100 -> Group_2 (Prob: 0.7760)
Rank 23/100 -> Group_2 (Prob: 0.7760)
Rank 24/100 -> Group_1 (Prob: 0.7759)
Rank 25/100 -> Group_2 (Prob: 0.7752)
Rank 26/100 -> Group_1 (Prob: 0.7730)
Rank 27/100 -> Group_3 (Prob: 0.7726)
Rank 28/100 -> Group_1 (Prob: 0.7725)
Rank 29/100 -> Group_2 (Prob: 0.7714)
Rank 30/100 -> Group_2 (Prob: 0.7711)
Rank 31/100 -> Group_2 (Prob: 0.7702)
Rank 32/100 -> Group_2 (Prob: 0.7699)
Rank 33/100 -> Group_1 (Prob: 0.7689)
Rank 34/100 -> Group_1 (Prob: 0.7689)
Rank 35/100 -> Group_2 (Prob: 0.7675)
Rank 36/100 -> Group_4 (Prob: 0.7675)
Rank 37/100 -> Group_2 (Prob: 0.7675)
Rank 38/100 -> Group_2 (Prob: 0.7673)
Rank 39/100 -> Group_2 (Prob: 0.7668)
Rank 40/100 -> Group_1 (Prob: 0.7661)
Rank 41/100 -> Group_2 (Prob: 0.7658)
Rank 42/100 -> Group_1 (Prob: 0.7654)
Rank 43/100 -> Group_3 (Prob: 0.7654)
Rank 44/100 